In [1]:
import tensorflow as tf

if tf.config.list_physical_devices('GPU'):
    print("GPU is available!")
    print(tf.config.list_physical_devices('GPU'))
else:
    print("GPU is not available. Please check runtime settings.")

GPU is not available. Please check runtime settings.


In [2]:
import os
import pandas as pd
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import joblib
from sklearn.utils import shuffle

In [3]:
# Đường dẫn đến dữ liệu trên Kaggle
base_dir = '/kaggle/input/cs114-all-cars/'
csv_dir = '/kaggle/input/split-data-car'
save_dir = '/kaggle/working/'

# Thay đổi đoạn code tải model
weights_path = '/kaggle/input/pretrained-weights/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5'
model = MobileNetV2(weights=weights_path, include_top=False, input_shape=(224, 224, 3))

# Bản đồ từ tên hiệu xe sang CategoryID
category_map = {
    "Others": 0,
    "Honda": 1,
    "Hyundai": 2,
    "KIA": 3,
    "Mazda": 4,
    "Mitsubishi": 5,
    "Suzuki": 6,
    "Toyota": 7,
    "VinFast": 8
}

In [4]:
# Trích xuất đặc trưng từ MobileNet
def extract_features(df, split_name):
    features_file = os.path.join(save_dir, f"Features_{split_name}_Split_{split_index}.npz")

    # Nếu tệp đã tồn tại, tải lại
    if os.path.exists(features_file):
        print(f"Loading features from {features_file}")
        data = np.load(features_file)
        return data['features'], data['labels']

    features, labels = [], []

    def process_image(row):
        try:
            image_path = os.path.join(base_dir, row['ImageFullPath'])
            label = category_map[row['ImageFullPath'].split('/')[0]]
            
            # Load và tiền xử lý ảnh
            image = load_img(image_path, target_size=(224, 224))
            image_array = img_to_array(image) / 255.0
            feature = model.predict(np.expand_dims(image_array, axis=0))
            features.append(feature.flatten())
            labels.append(label)
        except Exception as e:
            print(f"Error loading image {row['ImageFullPath']}: {e}")

    with ThreadPoolExecutor() as executor:
        executor.map(process_image, [row for _, row in df.iterrows()])

    # Lưu đặc trưng
    np.savez(features_file, features=np.array(features), labels=np.array(labels))
    print(f"Features saved to {features_file}")
    return np.array(features), np.array(labels)

In [5]:
# Số lượng split cần xử lý
num_splits = 5

# Giới hạn số mẫu cho từng batch
batch_size = 5000

# Lặp qua từng split
for split_index in range(1, num_splits + 1):
    if (split_index == 1):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split1-extract-features/Features_Train_Split_1.npz')
        features_file_test = os.path.join('/kaggle/input/split1-extract-features/Features_Test_Split_1.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    if (split_index == 2):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split23-extract-features/Features_Train_Split_2.npz')
        features_file_test = os.path.join('/kaggle/input/split23-extract-features/Features_Test_Split_2.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    if (split_index == 3):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split23-extract-features/Features_Train_Split_3.npz')
        features_file_test = os.path.join('/kaggle/input/split23-extract-features/Features_Test_Split_3.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    if (split_index == 4):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split45-extract-features/Features_Train_Split_4.npz')
        features_file_test = os.path.join('/kaggle/input/split45-extract-features/Features_Test_Split_4.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    if (split_index == 5):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split45-extract-features/Features_Train_Split_5.npz')
        features_file_test = os.path.join('/kaggle/input/split45-extract-features/Features_Test_Split_5.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    

    # train_path = os.path.join(csv_dir, f"CarDataset-Splits-{split_index}-Train.csv")
    # test_path = os.path.join(csv_dir, f"CarDataset-Splits-{split_index}-Test.csv")

    # # Đọc dữ liệu Train và Test
    # train_df = pd.read_csv(train_path)
    # test_df = pd.read_csv(test_path)

    # X_train, y_train = extract_features(train_df, "Train")
    # X_test, y_test = extract_features(test_df, "Test")

    # Huấn luyện với Mini-Batch
    classifier = SVC()

    # Shuffle dữ liệu
    indices = np.arange(len(X_train))
    np.random.shuffle(indices)
    X_train = X_train[indices]
    y_train = y_train[indices]

    # Huấn luyện từng batch
    num_samples = len(X_train)
    for start in range(0, num_samples, batch_size):
        end = min(start + batch_size, num_samples)
        X_batch, y_batch = X_train[start:end], y_train[start:end]
        classifier.fit(X_batch, y_batch)
        print(f"Split {split_index}: Trained on batch {start // batch_size + 1}")

    # Đánh giá trên tập test
    accuracy = classifier.score(X_test, y_test)
    y_pred = classifier.predict(X_test)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)

    # In kết quả
    print(f"Split {split_index}: Final Accuracy: {accuracy:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}")
    print(f"Classification Report:\n{class_report}")

    # Lưu kết quả vào file riêng cho từng split
    result_path = os.path.join(save_dir, f"Results_Split_{split_index}.txt")
    with open(result_path, 'w') as file:
        file.write(f"Split {split_index}: Final Accuracy: {accuracy:.4f}\n\n")
        file.write(f"Confusion Matrix:\n{conf_matrix}\n\n")
        file.write(f"Classification Report:\n{class_report}")

    # Lưu mô hình của từng split
    model_path = os.path.join(save_dir, f"Model_Split_{split_index}.joblib")
    joblib.dump(classifier, model_path)
    print(f"Split {split_index} model saved to {model_path}")

Split 1: Trained on batch 1
Split 1: Trained on batch 2
Split 1: Trained on batch 3
Split 1: Trained on batch 4
Split 1: Trained on batch 5
Split 1: Trained on batch 6
Split 1: Final Accuracy: 0.4107
Confusion Matrix:
[[555  16  58  11  31  17  86 121  14]
 [135 107  52  14  34  17  98 156  12]
 [176  13 186  22  32  11 119 124   2]
 [161  12  40 173  18   9 116  98  11]
 [208  13  37  13 174   8  66 100  15]
 [166  13  39  14  10  68 117 136   9]
 [ 92   9  25   7  15  20 947 186  10]
 [160  20  31   7  47  25 273 576  15]
 [194  12  13  15  19  19  90  75 126]]
Classification Report:
              precision    recall  f1-score   support

           0       0.30      0.61      0.40       909
           1       0.50      0.17      0.25       625
           2       0.39      0.27      0.32       685
           3       0.63      0.27      0.38       638
           4       0.46      0.27      0.34       634
           5       0.35      0.12      0.18       572
           6       0.50     

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Split 3: Final Accuracy: 0.3635
Confusion Matrix:
[[480   8  18   2  46   0 110 242]
 [147  51  10   4  27   0  92 294]
 [253   7  64   0  26   0 143 192]
 [171  10  23 142  20   0 104 167]
 [239   9   7   0 105   0  64 210]
 [186   7  10   4  13   0 129 223]
 [149   3   6   0  13   0 859 280]
 [175   5   6   0  30   0 268 670]]
Classification Report:
              precision    recall  f1-score   support

           0       0.27      0.53      0.35       906
           1       0.51      0.08      0.14       625
           2       0.44      0.09      0.15       685
           3       0.93      0.22      0.36       637
           4       0.38      0.17      0.23       634
           5       0.00      0.00      0.00       572
           6       0.49      0.66      0.56      1310
           7       0.29      0.58      0.39      1154

    accuracy                           0.36      6523
   macro avg       0.41      0.29      0.27      6523
weighted avg       0.41      0.36      0.32      6